# 04 Benchmark: `bank_loan_data.csv`

A **secondary benchmark only**. This notebook asks whether the patterns found in the primary
dataset (`loan_approval_dataset.csv`, which powers the app) also appear in a larger public
dataset, and which extra fields a real system would need.

Ground rules (CLAUDE.md):

- The target is the loan **approval decision**, not default. LoanLens never predicts default.
- This dataset is **never merged** with the primary one, and **nothing here changes the production
  model** in `models/` (this notebook does not import the training or saving functions).
- **Gender and age are never used as features** (sensitive attributes).
- Charts use the shared LoanLens theme: Approved green, Rejected red, neutral blue.

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

from loanlens import charts, theme
from loanlens import model as m
from loanlens.config import PROJECT_ROOT

pio.renderers.default = "plotly_mimetype+notebook_connected"
theme.register_plotly_template()
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.width", 160)
SEED = m.SEED  # 42, the same seed as the primary modelling notebook

RAW = PROJECT_ROOT / "data" / "raw" / "bank_loan_data.csv"
raw = pd.read_csv(RAW)
print(f"{len(raw):,} rows x {raw.shape[1]} columns")

45,000 rows x 14 columns


## 1. Profile

In [2]:
profile = pd.DataFrame(
    {
        "dtype": raw.dtypes.astype(str),
        "missing": raw.isna().sum(),
        "unique": raw.nunique(),
        "min": raw.min(numeric_only=True),
        "median": raw.median(numeric_only=True),
        "max": raw.max(numeric_only=True),
    }
)
profile

,dtype,missing,unique,min,median,max
cb_person_cred_hist_length,float64,0,29,2.0000,4.0000,30.0000
credit_score,int64,0,340,390.0000,640.0000,850.0000
loan_amnt,float64,0,4483,500.0000,"8,000.0000","35,000.0000"
loan_int_rate,float64,0,1302,5.4200,11.0100,20.0000
loan_intent,object,0,6,NaN,NaN,NaN
loan_percent_income,float64,0,64,0.0000,0.1200,0.6600
loan_status,int64,0,2,0.0000,0.0000,1.0000
person_age,float64,0,60,20.0000,26.0000,144.0000
person_education,object,0,5,NaN,NaN,NaN
person_emp_exp,int64,0,63,0.0000,4.0000,125.0000


In [3]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Class balance (loan_status):")
print(raw["loan_status"].value_counts().rename({1: "1 (approved)", 0: "0 (rejected)"}))
print(f"Approval share: {raw['loan_status'].mean():.1%}")
for col in raw.select_dtypes("object"):
    print(f"\n{col}: {raw[col].value_counts().to_dict()}")

Exact duplicate rows: 0
Class balance (loan_status):
loan_status
0 (rejected)    35000
1 (approved)    10000
Name: count, dtype: int64
Approval share: 22.2%

person_gender: {'male': 24841, 'female': 20159}

person_education: {'Bachelor': 13399, 'Associate': 12028, 'High School': 11972, 'Master': 6980, 'Doctorate': 621}

person_home_ownership: {'RENT': 23443, 'MORTGAGE': 18489, 'OWN': 2951, 'OTHER': 117}

loan_intent: {'EDUCATION': 9153, 'MEDICAL': 8548, 'VENTURE': 7819, 'PERSONAL': 7552, 'DEBTCONSOLIDATION': 7145, 'HOMEIMPROVEMENT': 4783}

previous_loan_defaults_on_file: {'Yes': 22858, 'No': 22142}


In [4]:
checks = {
    "age above 100": int((raw["person_age"] > 100).sum()),
    "employment experience above 60 years": int((raw["person_emp_exp"] > 60).sum()),
    "experience longer than (age - 14)": int(
        (raw["person_emp_exp"] > raw["person_age"] - 14).sum()
    ),
    "loan_percent_income != loan / income (by > 0.01)": int(
        ((raw["loan_amnt"] / raw["person_income"] - raw["loan_percent_income"]).abs() > 0.01).sum()
    ),
    "credit_score below 300 or above 850": int(
        ((raw["credit_score"] < 300) | (raw["credit_score"] > 850)).sum()
    ),
}
pd.Series(checks, name="rows")

age above 100                                        7
employment experience above 60 years                10
experience longer than (age - 14)                    0
loan_percent_income != loan / income (by > 0.01)     0
credit_score below 300 or above 850                  0
Name: rows, dtype: int64

In [5]:
by_defaults = raw.groupby("previous_loan_defaults_on_file")["loan_status"].agg(["mean", "size"])
by_gender = raw.groupby("person_gender")["loan_status"].agg(["mean", "size"])
print(by_defaults, "\n")
print(by_gender)

                                 mean   size
previous_loan_defaults_on_file              
No                             0.4516  22142
Yes                            0.0000  22858 

                mean   size
person_gender              
female        0.2225  20159
male          0.2220  24841


### Profile findings

- **Size and quality:** 45,000 rows, 14 columns, **no missing values** and no exact duplicates.
- **Class balance:** about **22% approved** (exactly 10,000 approved and 35,000 rejected, a
  round split typical of a generated dataset), against 62% in the primary
  dataset, so class-imbalance handling is needed (class weights below) and PR-AUC matters more
  than accuracy.
- **Units and currency:** no currency is stated. Incomes (median about 67,000, maximum 7.2
  million) and loan amounts (500 to 35,000) look like **US dollars**, not rupees, and are
  roughly 100 times smaller than the primary dataset's INR amounts. `loan_percent_income` is
  loan / annual income (verified above). `loan_int_rate` is a percentage.
- **Credit score scale:** 390 to 850, a **FICO-style 300-850 scale**, not CIBIL's 300-900. Scores
  cannot be compared one-to-one between the two datasets.
- **Signs that it is synthetic:** impossible values (ages up to 144, employment experience up
  to 125 years), and approval rates almost identical across genders and education levels.
  The public dataset card describes it as a synthetic, oversampled extension of an older
  credit-risk dataset.
- **Sensitive attributes:** `person_gender` and `person_age` are present. Neither is used as a
  feature (age is excluded as a sensitive attribute too).
- **A hard rule hidden in the data:** every applicant with a previous loan default on file is
  rejected (0% approval, 22,858 rows).
- **Label caution:** the dataset card says `loan_status` 1 = approved. But approval here *rises*
  with interest rate, loan-to-income and renting, the opposite of sound lending logic, and the
  same direction as the default label of the credit-risk dataset this one was derived from.
  The label may therefore mix approval and default semantics. It is treated here as an
  approval decision, as documented, but **no real-world lesson should be drawn from those
  directions**.

## 2. Cleaning

In [6]:
def clean_benchmark(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop impossible rows, map the defaults flag to 0/1, drop sensitive columns."""
    out = frame.copy()
    impossible = (out["person_age"] > 100) | (out["person_emp_exp"] > out["person_age"] - 14)
    out = out.loc[~impossible].copy()
    out["previous_defaults"] = (out["previous_loan_defaults_on_file"] == "Yes").astype(int)
    return out.drop(columns=["previous_loan_defaults_on_file", "person_gender", "person_age"])


bench = clean_benchmark(raw)
print(f"Dropped {len(raw) - len(bench)} impossible rows; {len(bench):,} rows remain.")
print("Sensitive columns left:", {"person_gender", "person_age"} & set(bench.columns) or "none")

Dropped 7 impossible rows; 44,993 rows remain.
Sensitive columns left: none


Only the 7 rows with an age above 100 are dropped (the experience check flags none). Ages
and gender are dropped from the frame entirely so no later cell can use them by accident.

## 3. EDA (approval decision, not default)

In [7]:
score_band = pd.cut(
    bench["credit_score"], [0, 550, 600, 650, 700, 850],
    labels=["<=550", "551-600", "601-650", "651-700", "701-850"],
).rename("credit_score_band")  # fmt: skip
charts.approval_rate_bar(bench, score_band, "Approval rate by credit score band", "Credit score")

In [8]:
charts.distribution_by_status(bench, "credit_score", "Credit score by decision")

In [9]:
defaults = bench["previous_defaults"].map({1: "Yes", 0: "No"}).rename("previous_defaults")
charts.approval_rate_bar(bench, defaults, "Approval rate by previous loan default on file")

In [10]:
charts.approval_heatmap(
    bench, score_band, defaults, "Approval rate: credit score band x previous default"
)

In [11]:
charts.approval_by_quantile_band(
    bench, "loan_percent_income", q=5, title="Approval rate by loan as a share of income"
)

In [12]:
charts.approval_by_quantile_band(
    bench, "loan_int_rate", q=5, title="Approval rate by interest rate (%)"
)

In [13]:
for col in ["person_home_ownership", "loan_intent", "person_education"]:
    print(bench.groupby(col)["loan_status"].agg(["mean", "size"]).round(3), "\n")

                        mean   size
person_home_ownership              
MORTGAGE              0.1160  18485
OTHER                 0.3330    117
OWN                   0.0750   2951
RENT                  0.3240  23440 

                    mean  size
loan_intent                   
DEBTCONSOLIDATION 0.3030  7145
EDUCATION         0.1700  9151
HOMEIMPROVEMENT   0.2630  4783
MEDICAL           0.2780  8548
PERSONAL          0.2010  7551
VENTURE           0.1440  7815 

                   mean   size
person_education              
Associate        0.2200  12026
Bachelor         0.2250  13396
Doctorate        0.2290    621
High School      0.2230  11970
Master           0.2180   6980 



### EDA findings

- **Credit score has almost no effect here:** approval is about 21-23% in every band. This is
  the opposite of the primary dataset, where CIBIL dominates (about 99% approval above 550
  and about 10% at or below).
- **Previous loan defaults decide everything they touch:** 0% approval with a default on file,
  about 45% without one.
- Approval **rises** with loan-to-income (about 11% in the lowest fifth to 55% in the highest)
  and with interest rate (about 9% to 49%). Renters are approved more often than owners.
  As noted in the profile, these directions contradict lending logic and look like a label
  artefact.
- Education has no effect (22-23% everywhere), as in the primary dataset.

## 4. Models: logistic regression, random forest, XGBoost

Same algorithms and settings as `03_modelling` (stratified 5-fold CV, seed 42, every scaler
and encoder fitted inside each fold through a Pipeline), plus **class-imbalance handling**:
`class_weight="balanced"` for logistic regression and random forest, `scale_pos_weight`
(rejected / approved) for XGBoost. The threshold stays at 0.5.

In [14]:
CATEGORICAL = ["person_education", "person_home_ownership", "loan_intent"]
NUMERIC = [
    "person_income", "person_emp_exp", "loan_amnt", "loan_int_rate", "loan_percent_income",
    "cb_person_cred_hist_length", "credit_score", "previous_defaults",
]  # fmt: skip
FEATURES = NUMERIC + CATEGORICAL
assert not {"person_gender", "person_age"} & set(FEATURES)
y = bench["loan_status"]
POS_WEIGHT = float((y == 0).sum() / (y == 1).sum())


def make_benchmark(kind: str, cols: list[str]) -> Pipeline:
    """Preprocessing + classifier with class-imbalance handling."""
    cat = [c for c in cols if c in CATEGORICAL]
    num = [c for c in cols if c not in CATEGORICAL]
    pre = ColumnTransformer(
        [
            ("num", StandardScaler() if kind == "logreg" else "passthrough", num),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ],
        verbose_feature_names_out=False,
        sparse_threshold=0,
    ).set_output(transform="pandas")
    if kind == "logreg":
        clf = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED)
    elif kind == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced",
            random_state=SEED, n_jobs=-1,
        )  # fmt: skip
    else:
        clf = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.1, eval_metric="logloss",
            scale_pos_weight=POS_WEIGHT, random_state=SEED, n_jobs=-1,
        )  # fmt: skip
    return Pipeline([("pre", pre), ("clf", clf)])


results = [
    m.cross_validate(kind, make_benchmark(kind, FEATURES), bench[FEATURES], y)
    for kind in ["logreg", "random_forest", "xgboost"]
]
comparison = m.summary_table(results)[
    ["roc_auc", "pr_auc", "precision", "recall", "f1", "accuracy", "f1_std"]
]
comparison

,roc_auc,pr_auc,precision,recall,f1,accuracy,f1_std
model,,,,,,,
logreg,0.9540,0.8541,0.6204,0.9201,0.7411,0.8570,0.0085
random_forest,0.9743,0.9288,0.8574,0.8095,0.8327,0.9277,0.0072
xgboost,0.9782,0.9383,0.7339,0.9199,0.8164,0.9080,0.0076


In [15]:
for r in results:
    cm = pd.DataFrame(
        r.confusion,
        index=["actually rejected", "actually approved"],
        columns=["predicted rejected", "predicted approved"],
    )
    print(r.name, "\n", cm, "\n")

logreg 
                    predicted rejected  predicted approved
actually rejected               29359                5634
actually approved                 799                9201 

random_forest 
                    predicted rejected  predicted approved
actually rejected               33646                1347
actually approved                1905                8095 

xgboost 
                    predicted rejected  predicted approved
actually rejected               31656                3337
actually approved                 801                9199 



With class weights the models trade precision for recall on the minority (approved) class.
Random forest has the best F1 and precision; XGBoost the best ROC-AUC and PR-AUC. Scores are
high but clearly below the primary dataset's near-perfect ones (ROC-AUC 0.994 for the
production model, about 0.999 unconstrained).

## 5. Ablations (XGBoost): which features dominate?

In [16]:
ablations = {
    "all features": FEATURES,
    "without credit_score": [c for c in FEATURES if c != "credit_score"],
    "without previous_defaults": [c for c in FEATURES if c != "previous_defaults"],
    "without credit_score and previous_defaults": [
        c for c in FEATURES if c not in ("credit_score", "previous_defaults")
    ],
    "previous_defaults only": ["previous_defaults"],
    "credit_score only": ["credit_score"],
    "without loan_int_rate": [c for c in FEATURES if c != "loan_int_rate"],
    "without loan_int_rate and loan_percent_income": [
        c for c in FEATURES if c not in ("loan_int_rate", "loan_percent_income")
    ],
}
ablation_results = [
    m.cross_validate(name, make_benchmark("xgboost", cols), bench[cols], y)
    for name, cols in ablations.items()
]
ablation_table = m.summary_table(ablation_results)[["roc_auc", "pr_auc", "f1"]]
ablation_table

,roc_auc,pr_auc,f1
model,,,
all features,0.9782,0.9383,0.8164
without credit_score,0.9773,0.9359,0.8116
without previous_defaults,0.9354,0.8742,0.7604
without credit_score and previous_defaults,0.9363,0.8752,0.7596
previous_defaults only,0.8266,0.4518,0.6224
credit_score only,0.5011,0.2242,0.3087
without loan_int_rate,0.9616,0.8930,0.7515
without loan_int_rate and loan_percent_income,0.9602,0.8870,0.7491


### Ablation findings

- **Removing credit score changes almost nothing** (ROC-AUC 0.978 to 0.977), and credit score
  alone is no better than chance (about 0.50).
- **Removing previous defaults costs the most** (ROC-AUC 0.978 to 0.935, F1 0.82 to 0.76).
  Previous defaults alone reach ROC-AUC 0.83.
- The rest of the signal is spread over income, interest rate, loan purpose, home ownership
  and loan-to-income (removing interest rate and loan-to-income together drops ROC-AUC to
  about 0.96).
- So the dominant feature differs completely between the datasets: **CIBIL in the primary
  data, previous defaults here**. Both are synthetic, so neither tells us how a real lender's
  real decisions work, but together they show that credit history fields matter and that one
  rule-like field can dominate a synthetic label.

## 6. SHAP (XGBoost, all features)

In [17]:
final = make_benchmark("xgboost", FEATURES).fit(bench[FEATURES], y)
X_enc = final[:-1].transform(bench[FEATURES])
values = shap.TreeExplainer(final[-1]).shap_values(X_enc)
per_column = pd.Series(np.abs(values).mean(axis=0), index=X_enc.columns)
# Sum the one-hot columns back to their original feature so shares are per input field.
owner = {c: next((f for f in CATEGORICAL if c.startswith(f + "_")), c) for c in X_enc.columns}
shap_share = per_column.groupby(owner).sum()
shap_share = (shap_share / shap_share.sum()).sort_values(ascending=False)
shap_share.rename("mean |SHAP| share")

previous_defaults            0.5808
person_income                0.0800
loan_int_rate                0.0764
loan_intent                  0.0727
person_home_ownership        0.0674
loan_percent_income          0.0672
credit_score                 0.0270
loan_amnt                    0.0154
person_emp_exp               0.0069
cb_person_cred_hist_length   0.0044
person_education             0.0018
Name: mean |SHAP| share, dtype: float32

In [18]:
charts.feature_importance_bar(
    shap_share.rename(index=lambda s: s.replace("_", " ")).head(10),
    "Benchmark: what drives the model (mean absolute SHAP share)",
)

Mean absolute SHAP share is the same measure the app's Analytics page uses for the production
model, so the two are directly comparable: there CIBIL has the largest share (57%); here
previous defaults have 58% and credit score only about 3%.

This model is fitted only to measure feature influence. It is **not saved** and does not touch
`models/`.

## 7. Conclusions

| | Primary (`loan_approval_dataset.csv`) | Benchmark (`bank_loan_data.csv`) |
|---|---|---|
| Rows | 4,269 | 45,000 (44,993 after cleaning) |
| Approved | 62% | 22% |
| Currency | INR | not stated, looks like USD |
| Credit score | CIBIL 300-900 | FICO-style, 390-850 observed |
| Dominant feature | CIBIL score | previous loan defaults |
| Credit score effect | decisive | none |
| Best ROC-AUC (5-fold CV) | about 0.999 (unconstrained) | about 0.978 (XGBoost) |

- Both datasets are synthetic and dominated by one rule-like field, which is why their scores
  are so high. Neither result is evidence of real-world accuracy.
- The benchmark has fields a real system should collect that the primary data lacks:
  **previous defaults, employment experience, home ownership, loan purpose, interest rate and
  credit-history length**.
- The production model and app are unchanged by this notebook. The full comparison is in
  `reports/benchmark.md`.